# **Regression**

# Predicting Attraction Rating

# 1. Import libraries

In [1]:
import pandas as pd

master = pd.read_excel("../data/cleaned/master_dataset.xlsx")
print(master.columns.tolist())
print(master.dtypes)

['TransactionId', 'UserId', 'VisitYear', 'VisitMonth', 'AttractionId', 'Rating', 'VisitMode', 'AttractionCityId', 'AttractionTypeId', 'Attraction', 'AttractionAddress', 'AttractionType', 'ContinentId', 'RegionId', 'CountryId', 'CityId', 'Continent', 'Region', 'Country', 'CityName']
TransactionId        int64
UserId               int64
VisitYear            int64
VisitMonth           int64
AttractionId         int64
Rating               int64
VisitMode              str
AttractionCityId     int64
AttractionTypeId     int64
Attraction             str
AttractionAddress      str
AttractionType         str
ContinentId          int64
RegionId             int64
CountryId            int64
CityId               int64
Continent              str
Region                 str
Country                str
CityName               str
dtype: object


# 2. Select features and target

In [2]:
# Select our features (inputs) and target (what we're predicting)
features = ['VisitMode', 'AttractionType', 'Continent', 'Region', 'Country', 'VisitYear', 'VisitMonth']
target = 'Rating'

X = master[features]
y = master[target]

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print(X.head())

Features shape: (52930, 7)
Target shape: (52930,)
  VisitMode           AttractionType Continent            Region  \
0   Couples  Nature & Wildlife Areas    Europe    Western Europe   
1   Friends  Nature & Wildlife Areas   America  Northern America   
2    Family  Nature & Wildlife Areas   America     South America   
3    Family  Nature & Wildlife Areas    Europe    Central Europe   
4   Couples  Nature & Wildlife Areas    Europe    Western Europe   

          Country  VisitYear  VisitMonth  
0  United Kingdom       2022          10  
1          Canada       2022          10  
2          Brazil       2022          10  
3     Switzerland       2022          10  
4  United Kingdom       2022          10  


# 3.Encode categorical columns

In [3]:
# One-hot encode all categorical (text) columns
X_encoded = pd.get_dummies(X, columns=['VisitMode', 'AttractionType', 'Continent', 'Region', 'Country'])

print("Shape before encoding:", X.shape)
print("Shape after encoding:", X_encoded.shape)
print(X_encoded.head())

Shape before encoding: (52930, 7)
Shape after encoding: (52930, 204)
   VisitYear  VisitMonth  VisitMode_Business  VisitMode_Couples  \
0       2022          10               False               True   
1       2022          10               False              False   
2       2022          10               False              False   
3       2022          10               False              False   
4       2022          10               False               True   

   VisitMode_Family  VisitMode_Friends  VisitMode_Solo  \
0             False              False           False   
1             False               True           False   
2              True              False           False   
3              True              False           False   
4             False              False           False   

   AttractionType_Ancient Ruins  AttractionType_Ballets  \
0                         False                   False   
1                         False                   False   
2 

# 4.Split into training and testing sets

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

Training set: (42344, 204)
Testing set: (10586, 204)


# 5. Train a baseline regression model

In [5]:
from sklearn.linear_model import LinearRegression

# Create and train the model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

print("Model trained!")

Model trained!


# 6. Make predictions and evaluate

In [6]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Make predictions on the test set
y_pred = lr_model.predict(X_test)

# Calculate evaluation metrics
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MSE (Mean Squared Error): {mse:.4f}")
print(f"RMSE (Root Mean Squared Error): {rmse:.4f}")
print(f"MAE (Mean Absolute Error): {mae:.4f}")
print(f"R² Score: {r2:.4f}")

MSE (Mean Squared Error): 0.8696
RMSE (Root Mean Squared Error): 0.9325
MAE (Mean Absolute Error): 0.7271
R² Score: 0.0767


# 7. stronger model — Random Forest Regressor

In [7]:
from sklearn.ensemble import RandomForestRegressor

# Create and train a Random Forest model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

print("Random Forest trained!")

Random Forest trained!


# 8. Evaluate Random Forest

In [8]:
# Make predictions on the test set
y_pred_rf = rf_model.predict(X_test)

# Calculate the same metrics as before
mse_rf = mean_squared_error(y_test, y_pred_rf)
rmse_rf = mse_rf ** 0.5
mae_rf = mean_absolute_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print(f"MSE: {mse_rf:.4f}")
print(f"RMSE: {rmse_rf:.4f}")
print(f"MAE: {mae_rf:.4f}")
print(f"R² Score: {r2_rf:.4f}")

# Quick side-by-side comparison with Linear Regression
print("\n--- Comparison ---")
print(f"{'Metric':<10} {'Linear Reg':<12} {'Random Forest':<12}")
print(f"{'RMSE':<10} {rmse:<12.4f} {rmse_rf:<12.4f}")
print(f"{'MAE':<10} {mae:<12.4f} {mae_rf:<12.4f}")
print(f"{'R²':<10} {r2:<12.4f} {r2_rf:<12.4f}")

MSE: 0.9671
RMSE: 0.9834
MAE: 0.7524
R² Score: -0.0268

--- Comparison ---
Metric     Linear Reg   Random Forest
RMSE       0.9325       0.9834      
MAE        0.7271       0.7524      
R²         0.0767       -0.0268     


# 9. Check for overfitting

In [9]:
# Compare performance on training data vs test data
train_pred = rf_model.predict(X_train)
train_r2 = r2_score(y_train, train_pred)

print(f"R² on Training data: {train_r2:.4f}")
print(f"R² on Test data: {r2_rf:.4f}")

R² on Training data: 0.4967
R² on Test data: -0.0268


# 10. Fix overfitting

In [10]:
# Retrain with constraints to prevent overfitting
rf_model_v2 = RandomForestRegressor(
    n_estimators=100,
    max_depth=8,          # limit how deep each tree can grow
    min_samples_leaf=20,  # each final "leaf" must represent at least 20 examples
    random_state=42,
    n_jobs=-1
)
rf_model_v2.fit(X_train, y_train)

# Evaluate on both train and test
train_pred_v2 = rf_model_v2.predict(X_train)
test_pred_v2 = rf_model_v2.predict(X_test)

train_r2_v2 = r2_score(y_train, train_pred_v2)
test_r2_v2 = r2_score(y_test, test_pred_v2)
test_rmse_v2 = mean_squared_error(y_test, test_pred_v2) ** 0.5
test_mae_v2 = mean_absolute_error(y_test, test_pred_v2)

print(f"R² on Training data: {train_r2_v2:.4f}")
print(f"R² on Test data: {test_r2_v2:.4f}")
print(f"RMSE on Test data: {test_rmse_v2:.4f}")
print(f"MAE on Test data: {test_mae_v2:.4f}")

R² on Training data: 0.1028
R² on Test data: 0.0955
RMSE on Test data: 0.9230
MAE on Test data: 0.7208


# 11. Save this as our final regression model

In [12]:
import joblib

joblib.dump(rf_model_v2, "../src/regression_model.pkl")
joblib.dump(X_encoded.columns.tolist(), "../src/regression_features.pkl")

print("Regression model saved!")

Regression model saved!
